In [1]:
import duckdb
con = duckdb.connect("agri_ghana.duckdb")

In [2]:
con.execute("SELECT DISTINCT unit FROM raw_wfp_prices").fetchdf()

,unit
0,50 KG
1,250 KG
2,20 KG
3,68 KG
4,16 KG
5,109 KG
6,93 KG
7,KG
8,27 KG
9,100 Tubers


In [3]:
con.execute("""
    SELECT 
        unit,
        price,
        CASE 
            WHEN unit = 'KG' THEN price
            WHEN unit LIKE '%KG' THEN price / CAST(regexp_extract(unit, '[0-9]+') AS DOUBLE)
        END AS price_per_kg
    FROM raw_wfp_prices
    WHERE unit = 'KG' OR unit LIKE '%KG'
    LIMIT 10
""").fetchdf()

,unit,price,price_per_kg
0,91 KG,10.31,0.113297
1,100 KG,25.40,0.254000
2,50 KG,33.06,0.661200
3,109 KG,38.25,0.350917
4,250 KG,63.38,0.253520
5,91 KG,16.00,0.175824
6,100 KG,16.15,0.161500
7,93 KG,32.50,0.349462
8,50 KG,21.56,0.431200
9,109 KG,34.50,0.316514


## SQL Staging Pipeline — Issue #1

**Goal:** Clean raw WFP price data and standardize inconsistent units into a single "price per 1 KG" baseline.

**Approach:**
- Rows with unit `KG` → price used as-is (already per-kg)
- Rows with unit like `50 KG`, `100 KG`, etc. → price divided by the number to get per-1kg price
- Rows with non-weight units (`Tubers`, `Bunch`, `pcs`) → **excluded**, since they can't be fairly converted to KG without knowing individual item weights

**Data quality check:** Among all weight-based (KG) rows, there were 0 missing prices and 0 missing dates — no imputation was necessary.

**Output:** `stg_wfp_prices` table in `agri_ghana.duckdb`

**Result:** 32,819 of 37,765 raw rows retained (~87%). The ~13% excluded were entirely due to non-weight units that cannot be standardized to a KG baseline.

In [4]:
con.execute("""
    CREATE OR REPLACE TABLE stg_wfp_prices AS
    SELECT 
        date,
        admin1,
        admin2,
        market,
        market_id,
        latitude,
        longitude,
        category,
        commodity,
        commodity_id,
        pricetype,
        currency,
        CASE 
            WHEN unit = 'KG' THEN price
            WHEN unit LIKE '%KG' THEN price / CAST(regexp_extract(unit, '[0-9]+') AS DOUBLE)
        END AS price_per_kg,
        usdprice
    FROM raw_wfp_prices
    WHERE unit = 'KG' OR unit LIKE '%KG'
      AND price IS NOT NULL
      AND date IS NOT NULL
""")

In [5]:
con.execute("SELECT COUNT(*) FROM stg_wfp_prices").fetchdf()

,count_star()
0,32819


**Result:** 32,819 of 37,765 raw rows retained (~87%).

The ~13% excluded were entirely due to non-weight units (Tubers, Bunch, pcs) that cannot be converted to a KG baseline. There were 0 rows with missing price among weight-based units, so no imputation was needed for this step.